In [21]:
import pandas as pd
import numpy as np

# processed dataset
df = pd.read_csv("processed_data/processed_data.csv")

### Split Data

In [26]:
def stratified_split(X, y, val_size=0.15, test_size=0.15, random_state=42):
    np.random.seed(random_state)

    # Combine X and y
    full_df = X.copy()
    full_df[target_col] = y

    # Shuffle samples
    full_df = full_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    # Split by class
    delayed = full_df[full_df[target_col] == 1]
    ontime = full_df[full_df[target_col] == 0]

    def split_class(df_class):
        n = len(df_class)
        test_n = int(n * test_size)
        val_n = int(n * val_size)
        test_set = df_class.iloc[:test_n]
        val_set = df_class.iloc[test_n:test_n + val_n]
        train_set = df_class.iloc[test_n + val_n:]
        return train_set, val_set, test_set

    train_1, val_1, test_1 = split_class(delayed)
    train_0, val_0, test_0 = split_class(ontime)

    train_df = pd.concat([train_1, train_0]).sample(frac=1, random_state=random_state)
    val_df = pd.concat([val_1, val_0]).sample(frac=1, random_state=random_state)
    test_df = pd.concat([test_1, test_0]).sample(frac=1, random_state=random_state)

    return train_df, val_df, test_df

# updated function
train_df, val_df, test_df = stratified_split(X, y)

# undersampling step on train_df
minority_class_size = train_df[target_col].sum()
train_df_majority = train_df[train_df[target_col] == 0].sample(n=minority_class_size, random_state=42)
train_df_minority = train_df[train_df[target_col] == 1]
train_df_balanced = pd.concat([train_df_majority, train_df_minority]).sample(frac=1, random_state=42)

# Final splits
X_train = train_df_balanced.drop(columns=[target_col])
y_train = train_df_balanced[target_col]

X_val = val_df.drop(columns=[target_col])
y_val = val_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

### Training, Validation, Test Sets

In [27]:
# check
print("Training class balance (after undersampling):\n", y_train.value_counts())
print("Validation set:\n", y_val.value_counts())
print("Test set:\n", y_test.value_counts())

Training class balance (after undersampling):
 Flight_Status_Binary
0    13853
1    13853
Name: count, dtype: int64
Validation set:
 Flight_Status_Binary
0    24871
1     2968
Name: count, dtype: int64
Test set:
 Flight_Status_Binary
0    24871
1     2968
Name: count, dtype: int64
